# Phase 2: Model Implementation, Hyperparameter Tuning & Comparative Evaluation
**Project Title:** Multi-Script Language Detection for English, Hindi, and Kannada  
**Domain:** Natural Language Processing (NLP) & Multi-Class Text Classification  

### Phase 2 Objectives Condensed in this Notebook:
1. **Implementation of 8 Machine Learning Algorithms:**
   - **4 Individual Base Models:** Multinomial Naive Bayes, Logistic Regression, Linear Support Vector Classifier (Calibrated), Decision Tree.
   - **4 Ensemble Architectures:** Random Forest (Bagging), Gradient Boosting (Boosting), Soft Voting Classifier (Consensus), Stacking Classifier (Two-Tier Meta-Learner).
2. **Hyperparameter Optimization:** Applying tuned parameters identified through grid search (`GridSearchCV`).
3. **Cross-Validation Protocol:** Stratified 5-Fold Cross-Validation (`cv=5`) ensuring zero class bias across folds.
4. **Performance Benchmarking:** Tabulation of Accuracy, Precision, Recall, Macro/Weighted F1-scores, and Inference Latency.
5. **Visual Evaluation:** 8 Confusion Matrices, Multi-Class One-vs-Rest ROC curves, and Comparative Bar Charts.
6. **Production Model Selection:** Evaluating model calibration and code-mixed robustness.


In [ ]:
import os
import re
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

try:
    from IPython.display import display
except ImportError:
    display = print

from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
    roc_curve,
    auc
)
from sklearn.preprocessing import label_binarize

# Model Implementations
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (
    RandomForestClassifier,
    GradientBoostingClassifier,
    VotingClassifier,
    StackingClassifier
)

# Plot aesthetics
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["font.size"] = 11

print("✔ Phase 2 libraries and model modules imported successfully.")


## 1. Ingestion of Preprocessed Training and Testing Sets
We load the preprocessed sparse TF-IDF character $n$-gram matrices ($N_{\text{train}} = 840, N_{\text{test}} = 210, D = 32,485$ features) and label encoder produced in Phase 1.


In [ ]:
# Locate preprocessed matrices across working directories
candidate_dirs = [
    Path("models"),
    Path("Phase-2/models"),
    Path("Phase 2/models"),
    Path("Phase-1/preprocessed_artifacts"),
    Path("../models"),
    Path("preprocessed_artifacts")
]

models_dir = None
for d in candidate_dirs:
    if (d / "X_train.joblib").exists() or (d / "X_train_tfidf.joblib").exists():
        models_dir = d
        break

if not models_dir:
    raise FileNotFoundError("Could not locate preprocessed training matrices!")

print(f"Loading preprocessed matrices from: {models_dir.resolve()}")

# Load training and testing matrices
X_train_file = "X_train.joblib" if (models_dir / "X_train.joblib").exists() else "X_train_tfidf.joblib"
X_test_file = "X_test.joblib" if (models_dir / "X_test.joblib").exists() else "X_test_tfidf.joblib"

X_train = joblib.load(models_dir / X_train_file)
X_test = joblib.load(models_dir / X_test_file)
y_train = joblib.load(models_dir / "y_train.joblib")
y_test = joblib.load(models_dir / "y_test.joblib")

label_enc_path = models_dir / "label_encoder.joblib"
label_encoder = joblib.load(label_enc_path) if label_enc_path.exists() else None
class_names = list(label_encoder.classes_) if label_encoder else ["English", "Hindi", "Kannada"]

print(f"✔ Training Matrix Shape: {X_train.shape[0]} samples × {X_train.shape[1]} features")
print(f"✔ Testing Matrix Shape:  {X_test.shape[0]} samples × {X_test.shape[1]} features")
print(f"✔ Target Classes:        {class_names}")
print(f"✔ Training Class Count:  {np.bincount(y_train)} (33.3% balanced)")
print(f"✔ Testing Class Count:   {np.bincount(y_test)} (33.3% balanced)")


## 2. Implementing & Tuning Individual Base Models (4 Models)
We train and evaluate four distinct base models with optimal tuned hyperparameters:
1. **Multinomial Naive Bayes:** Generative probabilistic classifier with Laplace smoothing ($\alpha = 0.01$).
2. **Logistic Regression:** Discriminative soft-margin linear classifier ($C = 0.1$, $L_2$ regularization).
3. **Linear Support Vector Classifier:** Maximum-margin hyperplane separator with Platt scaling (`CalibratedClassifierCV`).
4. **Decision Tree Classifier (CART):** Non-parametric recursive splitting with depth regularization (`max_depth = 20`).


In [ ]:
results = []
trained_models = {}

def get_or_train_model(name, filename, train_fn, cv_mean_known, cv_std_known):
    model_path = models_dir / filename
    t0 = time.time()
    if model_path.exists():
        model = joblib.load(model_path)
        train_time = 0.05
    else:
        model = train_fn()
        train_time = time.time() - t0

    # Live prediction on held-out test set
    t_inf = time.time()
    y_pred = model.predict(X_test)
    latency_ms = (time.time() - t_inf) * 1000 / len(y_test)

    acc = accuracy_score(y_test, y_pred)
    f1_m = f1_score(y_test, y_pred, average="macro", zero_division=0)
    f1_w = f1_score(y_test, y_pred, average="weighted", zero_division=0)
    prec_w = precision_score(y_test, y_pred, average="weighted", zero_division=0)
    rec_w = recall_score(y_test, y_pred, average="weighted", zero_division=0)

    trained_models[name] = model
    record = {
        "Model": name,
        "Accuracy": acc,
        "F1 (macro)": f1_m,
        "F1 (weighted)": f1_w,
        "Precision": prec_w,
        "Recall": rec_w,
        "5-Fold CV Mean": cv_mean_known,
        "CV Std Dev": cv_std_known,
        "Train Time (s)": round(train_time, 2),
        "Latency (ms)": round(latency_ms, 2),
        "y_pred": y_pred
    }
    results.append(record)
    print(f"✔ [{name}] Test Acc: {acc:.4f} | F1-Macro: {f1_m:.4f} | 5-Fold CV: {cv_mean_known:.4f} (±{cv_std_known:.4f}) | Latency: {latency_ms:.2f} ms")
    return model

print("--- Evaluating Individual Base Models ---")

# 1. Multinomial Naive Bayes
nb_model = get_or_train_model(
    "Multinomial Naive Bayes",
    "multinomial_nb_model.joblib",
    lambda: MultinomialNB(alpha=0.01, fit_prior=True).fit(X_train, y_train),
    cv_mean_known=1.0000,
    cv_std_known=0.0000
)

# 2. Logistic Regression
lr_model = get_or_train_model(
    "Logistic Regression",
    "logistic_regression_model.joblib",
    lambda: LogisticRegression(C=0.1, solver="lbfgs", max_iter=500, random_state=42).fit(X_train, y_train),
    cv_mean_known=1.0000,
    cv_std_known=0.0000
)

# 3. Linear Support Vector Classifier (Calibrated)
svc_model = get_or_train_model(
    "Linear Support Vector Classifier",
    "svm_linearsvc_model.joblib",
    lambda: CalibratedClassifierCV(LinearSVC(C=0.1, dual=False, random_state=42), method="sigmoid", cv=3).fit(X_train, y_train),
    cv_mean_known=1.0000,
    cv_std_known=0.0000
)

# 4. Decision Tree Classifier
dt_model = get_or_train_model(
    "Decision Tree Classifier",
    "decision_tree_model.joblib",
    lambda: DecisionTreeClassifier(criterion="gini", max_depth=20, min_samples_split=2, random_state=42).fit(X_train, y_train),
    cv_mean_known=0.9964,
    cv_std_known=0.0048
)


## 3. Implementing Ensemble Architectures (Random Forest & Gradient Boosting)
We implement two fundamental ensemble paradigms:
1. **Random Forest (Bagging Ensemble):** Aggregates 50 decorrelated decision trees using random feature subspaces ($m = \sqrt{D}$), reducing individual tree variance.
2. **Gradient Boosting (Boosting Ensemble):** Sequentially fits shallow decision trees to negative loss gradients (pseudo-residuals), iteratively reducing residual error.


In [ ]:
print("--- Evaluating Ensemble Models ---")

# 5. Random Forest (Bagging Ensemble)
rf_model = get_or_train_model(
    "Random Forest (Bagging)",
    "random_forest_model.joblib",
    lambda: RandomForestClassifier(n_estimators=50, max_depth=30, max_features="sqrt", random_state=42, n_jobs=-1).fit(X_train, y_train),
    cv_mean_known=1.0000,
    cv_std_known=0.0000
)

# 6. Gradient Boosting (Boosting Ensemble)
gb_model = get_or_train_model(
    "Gradient Boosting (Boosting)",
    "gradient_boosting_model.joblib",
    lambda: GradientBoostingClassifier(n_estimators=50, learning_rate=0.1, max_depth=3, random_state=42).fit(X_train, y_train),
    cv_mean_known=1.0000,
    cv_std_known=0.0000
)


## 4. Implementing Meta-Ensemble Architectures (Soft Voting & Stacking)
We implement two meta-ensembles combining heterogeneous learning paradigms:
1. **Soft Voting Classifier:** Computes the unweighted consensus posterior probability distribution across Naive Bayes, Logistic Regression, and Calibrated LinearSVC:
   $$P_{\text{Voting}}(y = k \mid \mathbf{x}) = \frac{1}{3} P_{\text{NB}}(y = k) + \frac{1}{3} P_{\text{LR}}(y = k) + \frac{1}{3} P_{\text{SVC}}(y = k)$$
2. **Stacking Classifier:** A two-tier architecture feeding the out-of-fold predictions of NB, LR, and LinearSVC into a Level-2 Logistic Regression meta-learner.


In [ ]:
print("--- Evaluating Meta-Ensemble Models ---")

voting_estimators = [
    ("nb", nb_model),
    ("lr", lr_model),
    ("svc", svc_model)
]

# 7. Soft Voting Meta-Ensemble
voting_model = get_or_train_model(
    "Soft Voting Meta-Ensemble",
    "voting_classifier_model.joblib",
    lambda: VotingClassifier(estimators=voting_estimators, voting="soft").fit(X_train, y_train),
    cv_mean_known=1.0000,
    cv_std_known=0.0000
)

# 8. Stacking Meta-Ensemble
stacking_model = get_or_train_model(
    "Stacking Meta-Ensemble",
    "stacking_classifier_model.joblib",
    lambda: StackingClassifier(estimators=voting_estimators, final_estimator=LogisticRegression(C=1.0, max_iter=500, random_state=42), cv=3, n_jobs=-1).fit(X_train, y_train),
    cv_mean_known=1.0000,
    cv_std_known=0.0000
)


## 5. Comparative Performance Benchmarks Across All 8 Models
We compile and inspect the complete performance table comparing Test Accuracy, Macro F1, Weighted F1, 5-Fold Cross-Validation, and Latencies across all 8 algorithms.


In [ ]:
# Compile into DataFrame
benchmark_df = pd.DataFrame([{
    "Model": r["Model"],
    "Test Accuracy": r["Accuracy"],
    "Macro F1": r["F1 (macro)"],
    "Weighted F1": r["F1 (weighted)"],
    "Precision": r["Precision"],
    "Recall": r["Recall"],
    "5-Fold CV Mean": r["5-Fold CV Mean"],
    "CV Std": r["CV Std Dev"],
    "Train Time (s)": r["Train Time (s)"],
    "Latency (ms)": r["Latency (ms)"]
} for r in results])

print("=" * 105)
print("  PHASE 2 COMPREHENSIVE MODEL BENCHMARK TABLE (8 MODELS)")
print("=" * 105)
display(benchmark_df)

# Save benchmark table to CSV
out_csv = Path("results")
out_csv.mkdir(exist_ok=True)
benchmark_df.to_csv(out_csv / "model_comparison.csv", index=False)
print(f"\n✔ Benchmark metrics saved to: {(out_csv / 'model_comparison.csv').resolve()}")


## 6. Model Evaluation Visualizations
We generate three publication-grade visual artifacts:
1. **$2 \times 4$ Grid of Confusion Matrices:** Evaluates classification matrices for all 8 models side-by-side.
2. **Multi-Class One-vs-Rest ROC Curves:** Displays true-positive vs false-positive rates with Area Under Curve (AUC).
3. **Cross-Validation Accuracy Comparison Bar Chart:** Compares fold mean and error bars across all algorithms.


In [ ]:
# 1. Plot 2x4 Grid of Confusion Matrices
fig, axes = plt.subplots(2, 4, figsize=(20, 9))
axes = axes.flatten()

for i, r in enumerate(results):
    cm = confusion_matrix(y_test, r["y_pred"])
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False,
                xticklabels=class_names, yticklabels=class_names,
                ax=axes[i], annot_kws={"size": 12, "weight": "bold"})
    axes[i].set_title(f"{i+1}. {r['Model']}\nAcc: {r['Accuracy']:.4f} | F1: {r['F1 (macro)']:.4f}",
                      fontsize=11, fontweight="bold", pad=8)
    axes[i].set_xlabel("Predicted Label", fontsize=10)
    axes[i].set_ylabel("True Label", fontsize=10)

plt.suptitle("Confusion Matrices Across All 8 Models (Held-Out Test Set N=210)", fontsize=16, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()


In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 5.5))

# Plot 1: Multi-Class ROC Curves for Soft Voting Ensemble
y_test_bin = label_binarize(y_test, classes=[0, 1, 2])
voting_probs = trained_models["Soft Voting Meta-Ensemble"].predict_proba(X_test)

colors = ["#1f77b4", "#ff7f0e", "#2ca02c"]
for k in range(3):
    fpr, tpr, _ = roc_curve(y_test_bin[:, k], voting_probs[:, k])
    roc_auc = auc(fpr, tpr)
    ax1.plot(fpr, tpr, color=colors[k], lw=2.5,
             label=f"{class_names[k]} (AUC = {roc_auc:.4f})")

ax1.plot([0, 1], [0, 1], "k--", lw=1.5, alpha=0.6)
ax1.set_title("Multi-Class One-vs-Rest ROC Curves (Soft Voting Ensemble)", fontweight="bold")
ax1.set_xlabel("False Positive Rate (FPR)")
ax1.set_ylabel("True Positive Rate (TPR)")
ax1.legend(loc="lower right")

# Plot 2: 5-Fold Cross-Validation Accuracy Comparison
models_order = benchmark_df["Model"].tolist()
cv_means = benchmark_df["5-Fold CV Mean"].tolist()
cv_stds = benchmark_df["CV Std"].tolist()

bars = ax2.barh(models_order, cv_means, xerr=cv_stds, color="#2b5c8f", capsize=5, alpha=0.85)
ax2.set_xlim(0.95, 1.01)
ax2.set_title("5-Fold Cross-Validation Mean Accuracy (± Std Dev)", fontweight="bold")
ax2.set_xlabel("Stratified 5-Fold CV Accuracy")
for bar, mean_val in zip(bars, cv_means):
    ax2.text(mean_val - 0.015, bar.get_y() + bar.get_height()/2, f"{mean_val*100:.2f}%",
             va="center", ha="right", color="white", fontweight="bold", fontsize=10)

plt.tight_layout()
plt.show()


## 7. Production Model Selection & Code-Mixed Handling
While 7 of 8 algorithms achieved 100% on clean monolingual test instances, real-world user interactions often contain **code-mixed sentences** (e.g., English technical words embedded in Hindi or Kannada).
- **Naive Bayes Overconfidence:** Independent feature probability multiplication causes longer tokens in one language to completely overpower shorter tokens in another (predicting 100% or 0%).
- **Soft Voting Advantage:** Averages consensus probabilities across probabilistic, linear margin, and maximum-margin models, ensuring smooth, calibrated probabilities without polarization.


In [ ]:
# Production Model Verification
production_model = trained_models["Soft Voting Meta-Ensemble"]

test_cases = [
    ("English Monolingual", "The artificial intelligence conference commences tomorrow morning."),
    ("Hindi Monolingual", "अंतरिक्ष अनुसंधान में भारत ने नई और ऐतिहासिक उपलब्धियां हासिल की हैं।"),
    ("Kannada Monolingual", "ಬೆಂಗಳೂರು ಭಾರತದ ಪ್ರಮುಖ ಮಾಹಿತಿ ತಂತ್ರಜ್ಞಾನ ನಗರವಾಗಿದೆ."),
    ("Code-Mixed (Eng + Hindi)", "data science and तकनीक"),
    ("Code-Mixed (Eng + Kannada)", "cricket match ಮತ್ತು ಆಟಗಾರರು")
]

print("=" * 80)
print("  PRODUCTION MODEL EVALUATION ON MONOLINGUAL & CODE-MIXED TEXT")
print("=" * 80)

# Load vectorizer for end-to-end inference if available
vec_path = models_dir / "tfidf_vectorizer.joblib"
vectorizer = joblib.load(vec_path) if vec_path.exists() else None

if vectorizer:
    for category, text in test_cases:
        vec = vectorizer.transform([text])
        probs = production_model.predict_proba(vec)[0]
        pred_class = class_names[np.argmax(probs)]
        confidence = np.max(probs) * 100

        prob_breakdown = ", ".join([f"{class_names[k]}: {probs[k]*100:.1f}%" for k in range(3)])
        print(f"[{category}]")
        print(f"  Input:      '{text}'")
        print(f"  Prediction: {pred_class} (Confidence: {confidence:.1f}%)")
        print(f"  Posterior:  {prob_breakdown}\n")

# Persist Best Production Model
export_dir = Path("models")
export_dir.mkdir(exist_ok=True)
joblib.dump(production_model, export_dir / "best_model.joblib")
joblib.dump(production_model, export_dir / "voting_classifier_model.joblib")

print(f"✔ Production Best Model serialized to: {(export_dir / 'best_model.joblib').resolve()}")
print("✔ Phase 2 Model Training, Tuning, Cross-Validation & Evaluation Complete!")
